# Module 1 — Foundations: Causal Inference & When to Test

Part of the **A/B Testing Playbook** learning program. Full program context, progress tracker, and how this module fits with the rest: see `CURRICULUM.md`.

## From the curriculum

> **Concepts**
> - Correlation vs. causation, and why RCTs (Randomized Controlled Trials — the formal name for what a product team calls an "A/B test") are the gold standard for causal claims
> - The counterfactual framing: what would have happened without the change
> - When A/B testing is the right tool vs. when it isn't (low traffic, ethical constraints, network effects, one-time launches, infrastructure changes)
> - The B2C experimentation "culture" — why companies run hundreds of concurrent tests, and what that implies about tooling and rigor
>
> **Why it matters**
> This is the mental model everything else hangs off. B2B analysts often reason causally from small-n, pre/post comparisons — B2C interviewers will probe whether you reach for a proper experiment instead of a correlation.
>
> **Worked example**
> A product team notices users who enable push notifications have 40% higher 30-day retention than users who don't, and wants to credit push notifications for the lift. The catch: users who *choose* to enable notifications are probably already more engaged — the comparison is confounded by self-selection, not causal. The fix is an A/B test that *randomly assigns* users to see a notification opt-in prompt (or not) — now the two groups differ only in what they were offered, so any retention gap is attributable to the prompt itself.
>
> **Resources**
> - *Trustworthy Online Controlled Experiments* — Kohavi, Tang, Xu (book). Chapter 1 only (~20 min) — the clearest short framing of why controlled experiments beat observational analysis.
>
> **Exercise type:** short case-based judgment calls ("would you A/B test this, and why/why not") rather than computation.
>
> **Interview angle:** "How would you test X?" / "When would you *not* run an A/B test?" — extremely common opener.

## Lesson

### Correlation vs. causation

Two variables move together for basically three reasons: (1) one causes the other, (2) something else (a **confounder**) causes both, or (3) coincidence. Observational data — the kind you pull from a warehouse without ever having *intervened* on anything — can't tell these apart on its own. A B2B analyst is used to this problem too (e.g. "accounts with an assigned CSM renew more") — the B2C twist is mostly that the confounders are individual-user behavioral traits (engagement, intent, tenure) rather than account-level ones, and there's enough traffic that you can usually just run the experiment instead of arguing about the confound.

### The counterfactual framing

The real object we want for any single user is: *what would have happened to this exact user under the treatment, versus what would have happened to this exact user without it?* That's the causal effect for that user. The catch — sometimes called the fundamental problem of causal inference — is that we only ever observe **one** of those two outcomes for any given person; they either got the treatment or they didn't. We can never directly see the road not taken for them individually.

What we *can* do is compare **groups** that are, in expectation, identical in every way except treatment — so the average outcome in the untreated group is a valid stand-in for "what would have happened to the treated group without treatment." That's exactly what random assignment buys you: it makes the treatment and control groups comparable not just on the confounders you thought to measure, but on the ones you didn't. That's the whole reason RCTs (randomized controlled trials — the formal name for what a product team calls an "A/B test") are the gold standard: you don't have to know or measure every confounder, you just have to randomize.

### When A/B testing is (and isn't) the right tool

A/B testing is the right default whenever you *can* randomize individual units and get a result in reasonable time. It stops being the right tool — or needs a workaround — when:

- **Traffic is too low** to reach adequate statistical power in a reasonable window (common in B2B: dozens or hundreds of accounts, not thousands of users/day)
- **Ethical or legal constraints** make withholding a treatment from a random half of users unacceptable (e.g. a safety fix, a legally mandated disclosure)
- **Network effects / interference** mean treated and control units interact and contaminate each other (e.g. a marketplace feature, a social feed change) — covered properly in Module 6
- **It's a one-time, all-or-nothing change** with no natural way to show two simultaneous variants (e.g. a full infrastructure migration, a company-wide pricing policy change with PR exposure)
- **The effect only shows up in aggregate/systemic terms**, not at the individual level (e.g. a change to how the marketplace clears)

For most of these, Module 8 (quasi-experiments) covers the fallback toolkit — diff-in-differences, switchback tests, geo experiments — for when true randomization isn't available but you still want more rigor than a plain before/after comparison.

### Why B2C leans so heavily on this machinery

At B2C scale, traffic is cheap and product velocity is high — a company might have hundreds of experiments running concurrently, each one a randomized comparison on live users. That volume is *why* the tooling and rigor around experimentation (Modules 3–7) exists: at that scale, small mistakes in design or interpretation compound across hundreds of decisions, and organizations invest in platforms specifically to keep every one of those experiments trustworthy. B2B doesn't build the same machinery mainly because it doesn't have the traffic to need it as often — not because the underlying causal-inference problem is any different.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)
n = 20_000

# Each user has a latent "engagement propensity" — how likely they are to
# stick around regardless of anything a product team does.
propensity = rng.beta(a=2, b=5, size=n)  # skewed toward lower engagement, some high-engagement users

# TRUE retention model: propensity drives retention. Notifications have
# NO causal effect on retention — the true effect we're simulating is zero.
true_effect = 0.0
retention_prob = np.clip(propensity + true_effect, 0, 1)
retained = rng.binomial(1, retention_prob)

df = pd.DataFrame({"propensity": propensity, "retained": retained})

### Scenario A — observational comparison (confounded)

Here, whether a user enabled notifications is *not* random — more engaged users (higher propensity) are more likely to opt in. This mirrors the curriculum's worked example.

In [2]:
# Notification opt-in is NOT random — it's driven by the same propensity
# that drives retention. This is the confound.
opt_in_prob = np.clip(propensity * 1.3, 0, 1)
df["notifications_enabled"] = rng.binomial(1, opt_in_prob)

naive = df.groupby("notifications_enabled")["retained"].mean()
naive_lift = naive[1] - naive[0]

print("Retention by notification status (observational, confounded):")
print(naive)
print(f"\nApparent lift: {naive_lift:+.1%}  <- looks like a real effect, but isn't one")

Retention by notification status (observational, confounded):
notifications_enabled
0    0.236545
1    0.368414
Name: retained, dtype: float64

Apparent lift: +13.2%  <- looks like a real effect, but isn't one


### Scenario B — randomized experiment (the fix)

Now assign the notification prompt **independently of propensity** — a true coin flip per user, exactly like a real A/B test. The true effect hasn't changed (still zero); only how we assigned the "treatment" has.

In [3]:
df["randomized_group"] = rng.integers(0, 2, size=n)  # 0 = control, 1 = treatment, independent of propensity

experiment = df.groupby("randomized_group")["retained"].mean()
experiment_lift = experiment[1] - experiment[0]

print("Retention by randomized group (true A/B test):")
print(experiment)
print(f"\nEstimated lift: {experiment_lift:+.1%}  <- close to the true effect (0%), as it should be")

Retention by randomized group (true A/B test):
randomized_group
0    0.285586
1    0.285614
Name: retained, dtype: float64

Estimated lift: +0.0%  <- close to the true effect (0%), as it should be


### What this shows

The observational comparison (Scenario A) shows a large apparent lift from notifications — driven entirely by the fact that engaged users were more likely to opt in, not by notifications doing anything. The randomized comparison (Scenario B), using the *same* underlying retention model with the *same* true zero effect, correctly shows roughly no difference between groups. Nothing about the world changed between the two scenarios — only whether assignment was confounded or random. That's the entire case for why A/B testing beats observational analysis: it's not that observational data is useless, it's that it can't rule out exactly this kind of story on its own.

(We're not running a formal significance test on this yet — "is that residual gap distinguishable from zero" is exactly the question Module 3 gives us the tools to answer properly.)

## External resource — read before the exercises below

**Must-read (~20 min):** *Trustworthy Online Controlled Experiments* by Kohavi, Tang & Xu — **Chapter 1** only. Free official PDF: [Chapter 1](https://experimentguide.com/wp-content/uploads/TrustworthyOnlineControlledExperiments_PracticalGuideToABTesting_Chapter1.pdf), via [experimentguide.com](https://experimentguide.com/), the book's official companion site. It's the clearest short framing available of why controlled experiments beat observational analysis, and everything above assumes you've seen that framing. This book is the spine resource for the whole program — later modules cite specific chapters, which need the full book (purchase links also on the companion site).

## Exercises — would you A/B test this?

For each scenario: would you A/B test it? Why or why not? If not, what would you reach for instead — a quasi-experiment (Module 8 has the toolkit), or something else? Write your answer in the empty markdown cell under each scenario.

**Scenario A.** You want to know if changing a checkout button's color from blue to green increases purchases. You have ~50,000 checkout sessions/day.

**Model answer (for comparison — write your own first if you haven't yet):**

<span style="color:#1a7f37"><b>Yes</b></span> — this is the textbook case. ~50,000 sessions/day means power is cheap (Module 3 will let us size this exactly), there's no ethical or network-effect issue, and it's a single, conclusive, individual-level decision. Randomize by **user** (not session) so the same person doesn't see both colors across visits — that consistency is a design detail from Module 4. Primary metric: purchase conversion rate. A guardrail worth watching: revenue per session (make sure any lift isn't coming from smaller/impulse purchases only).

**Scenario B.** Leadership wants to raise prices nationally by 5%, and wants to know the revenue impact before committing. The change requires legal/PR sign-off and can't be shown to only half of customers without causing visible, reportable inconsistency.

**Model answer:**

<span style="color:#cf222e"><b>No</b></span> — not as a plain individual-level A/B test. Two blockers: (1) charging different customers different prices for the same thing at the same time is a legal/PR exposure the scenario explicitly flags, not just a design inconvenience, and (2) this is a policy-level change, not something that naturally splits into "variant shown to this user." This is exactly the Module 8 situation: reach for a **quasi-experiment** — most likely a staggered geo rollout (raise prices in a handful of states first, keep demographically similar states as control) analyzed with difference-in-differences. It's weaker causally than a true RCT, but it's what's available given the constraint.

**Scenario C.** You want to know if a redesigned onboarding email increases activation. You get ~50,000 new signups per week.

**Model answer:**

<span style="color:#1a7f37"><b>Yes</b></span> — same shape as Scenario A. 50,000 signups/week is plenty of volume, it's a single well-defined user-level change (which onboarding email a new signup gets), and there's no ethical/network blocker. Randomize by user at signup. Primary metric: activation rate. Worth a guardrail on unsubscribe/spam-complaint rate, since a more aggressive email could lift short-term activation while damaging long-term trust.

**Scenario D.** Product wants to remove a feature entirely (not offer a variant — remove it) to see if complaint volume drops. The feature is used in shared/social contexts, so users who still had it would notice and discuss it with users who didn't.

**Model answer:**

<span style="color:#cf222e"><b>No</b></span> — not as a plain user-level test. This is the network-effect/interference trap flagged in the curriculum, covered properly in Module 6. Even if you randomly assign users to "keep feature" vs. "feature removed," it's a shared/social feature — control users will notice their treatment-group friends/counterparts no longer have it, which contaminates the comparison (control isn't a clean stand-in for "no change happened" anymore). A plain user-level A/B test isn't valid here. Better options: randomize at a **cluster level** (e.g. by social group or market, if the groups are reasonably separable) so contamination stays within-cluster rather than across arms, or fall back to a staggered rollout with before/after comparison against a held-out cluster (Module 8 territory) — imperfect, but more honest than pretending user-level randomization gives a clean answer here.

**Scenario E (B2B callback).** You have 40 enterprise accounts total and want to know if a new sales deck increases close rate.

**Model answer:**

<span style="color:#cf222e"><b>No</b></span> — 40 accounts is nowhere close to enough for a powered randomized test (Module 3 will make "nowhere close" precise, but the intuition holds already: you'd need an enormous effect size to detect anything reliably at n=40, split into two even smaller groups). This is the classic B2B constraint from the "why it matters" note in this module's lesson: not a different causal-inference problem, just not enough volume to solve it the same way B2C does. Realistic alternative: don't try to force statistical significance — track the outcome directionally over a longer window, lean on qualitative feedback from the sales reps using the new deck, and treat any observed difference as suggestive at best, not a validated causal claim.

## Key takeaways

- Correlation can't distinguish "X causes Y" from "a confounder causes both" — you need either randomization or a credible substitute (Module 8)
- The counterfactual is never directly observable for an individual; randomization makes group-level comparisons a valid stand-in
- A/B testing is the default at B2C scale because traffic makes randomization cheap — not because the underlying causal-inference problem is different in B2B, just harder to solve the same way
- Know the specific conditions that push you off plain A/B testing (low traffic, ethics/legal, network effects, one-off changes) — these come up directly in design-case interview questions

---
*Status: mark this module's row in `CURRICULUM.md`'s progress table once reviewed.*